In [1]:
import requests
from pprint import pprint
import time

In [2]:
import json
import os
import requests
import pandas as pd

API_KEY = "PAvN52bvt5UMEFhI9wiSG05pDiYSbK1j"  # Replace with your actual Ticketmaster API Key
BASE_URL = "https://app.ticketmaster.com/discovery/v2/events.json"

# UK Target Cities to bypass the 1000-record API cap per single query scope
TARGET_CITIES = [
    "London", "Manchester", "Birmingham", "Glasgow", "Leeds", 
    "Liverpool", "Bristol", "Sheffield", "Newcastle", "Nottingham"
]

# ONS Mid-Year Population Data Estimates
ONS_POPULATION_DATA = {
    "London": 8908000,
    "Birmingham": 1144000,
    "Glasgow": 635000,
    "Manchester": 553000,
    "Liverpool": 496000,
    "Bristol": 472000,
    "Sheffield": 584000,
    "Leeds": 812000,
    "Newcastle": 300000,
    "Nottingham": 323000
}

def fetch_raw_rock_events():
    """
    Queries Ticketmaster API for Rock music events across key UK cities.
    Writes raw response payloads to disk BEFORE processing.
    """
    os.makedirs("raw_data", exist_ok=True)
    all_raw_events = []

    print("=== STARTING LIVE DATA INGESTION ===")
    for city in TARGET_CITIES:
        page = 0
        page_size = 200
        
        while True:
            # API Guardrail: Enforce 1,000 record cap truncation check (page * size < 1000)
            if page * page_size >= 1000:
                print(f"⚠️ [Truncation Warning] City '{city}' hit 1,000 record cap on page {page}.")
                break

            params = {
                "apikey": API_KEY,
                "countryCode": "GB",
                "segmentName": "Music",       # Hierarchy: Segment
                "classificationName": "Rock",  # Hierarchy: Genre
                "city": city,
                "size": page_size,
                "page": page
            }

            response = requests.get(BASE_URL, params=params)
            
            # Rate limiting guardrail: sleep 0.25s (max 4 requests/sec, safely under 5 req/sec cap)
            time.sleep(0.25)

            if response.status_code != 200:
                print(f"Error fetching {city} (Page {page}): HTTP {response.status_code}")
                break

            data = response.json()
            events = data.get("_embedded", {}).get("events", [])
            
            # Tag each event payload with the query city to preserve mapping context
            for ev in events:
                ev["_query_city"] = city
                
            all_raw_events.extend(events)

            page_info = data.get("page", {})
            total_pages = page_info.get("totalPages", 1)
            total_elements = page_info.get("totalElements", 0)

            print(f"City: {city:12s} | Page {page + 1}/{total_pages} | Fetched: {len(events)} | Total Available: {total_elements}")

            page += 1
            if page >= total_pages:
                break

    # GUARDRAIL: Write untouched raw response payload to disk BEFORE parsing
    raw_file = os.path.join("raw_data", "raw_rock_events_all.json")
    with open(raw_file, "w", encoding="utf-8") as f:
        json.dump(all_raw_events, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Raw ingestion complete. Saved {len(all_raw_events)} raw events across {len(TARGET_CITIES)} cities to '{raw_file}'.")
    return all_raw_events


def parse_and_clean_data(raw_events):
    """
    Parses raw event JSON into a DataFrame.
    Strips city whitespace, checks null rates, and excludes incomplete classification/act records.
    """
    parsed = []

    for ev in raw_events:
        classifications = ev.get("classifications", [{}])[0]
        segment = classifications.get("segment", {}).get("name")
        genre = classifications.get("genre", {}).get("name")
        subgenre = classifications.get("subGenre", {}).get("name")

        venues = ev.get("_embedded", {}).get("venues", [{}])
        # Fall back to query city if venue city string is missing
        raw_city = venues[0].get("city", {}).get("name") if venues else ev.get("_query_city")
        venue_name = venues[0].get("name") if venues else None

        attractions = ev.get("_embedded", {}).get("attractions", [{}])
        attraction_id = attractions[0].get("id") if attractions else None
        attraction_name = attractions[0].get("name") if attractions else None

        prices = ev.get("priceRanges", [{}])
        min_price = prices[0].get("min") if prices else None
        max_price = prices[0].get("max") if prices else None

        parsed.append({
            "event_id": ev.get("id"),
            "event_name": ev.get("name"),
            "start_date": ev.get("dates", {}).get("start", {}).get("localDate"),
            "attraction_id": attraction_id,
            "act_name": attraction_name,
            "segment": segment,
            "genre": genre,
            "subgenre": subgenre,
            "city": raw_city,
            "venue": venue_name,
            "min_price": min_price,
            "max_price": max_price
        })

    df = pd.DataFrame(parsed)
    # CRITICAL FIX: Strip leading/trailing whitespace from city names to stop duplicate NaN rows
    if "city" in df.columns:
        df["city"] = df["city"].astype(str).str.strip()

    # GUARDRAIL: Check & report missingness prior to dropping/aggregating
    print("\n--- NULL RATE REPORT ---")
    for col in ["genre", "subgenre", "attraction_id", "min_price"]:
        null_count = df[col].isnull().sum()
        null_pct = (null_count / len(df)) * 100 if len(df) > 0 else 0
        print(f"Column '{col}': {null_count} missing ({null_pct:.1f}%)")

    # GUARDRAIL: Apply chosen Null Strategy -> EXCLUDE missing classification/act records
    # Note: min_price is kept optional due to high API omission rates
    before_count = len(df)
    df_cleaned = df.dropna(subset=["genre", "subgenre", "attraction_id"]).copy()
    print(f"Applied NULL Strategy 'Exclude': Dropped {before_count - len(df_cleaned)} incomplete classification/act records.")

    return df_cleaned


def compute_market_saturation(df):
    """
    Calculates market saturation using ONS Per-Capita Population baseline.
    Deduplicates on attraction_id to measure UNIQUE acts per market.
    """
    # GUARDRAIL: Act-level deduplication per city (1 artist playing 3 nights = 1 act)
    city_act_counts = (
        df.groupby("city")["attraction_id"]
        .nunique()
        .reset_index()
        .rename(columns={"attraction_id": "unique_acts"})
    )

    city_event_counts = df.groupby("city")["event_id"].count().reset_index().rename(columns={"event_id": "total_events"})
    
    summary = pd.merge(city_act_counts, city_event_counts, on="city")
    summary["population_ons"] = summary["city"].map(ONS_POPULATION_DATA)

    # Compute Saturation Metric: Unique Acts per 100k residents
    summary["acts_per_100k"] = (summary["unique_acts"] / summary["population_ons"]) * 100000
    summary = summary.sort_values(by="acts_per_100k", ascending=False).dropna(subset=["population_ons"])
    
    return summary


if __name__ == "__main__":
    raw_data = fetch_raw_rock_events()
    cleaned_df = parse_and_clean_data(raw_data)
    
    # Save cleaned output for Streamlit dashboard ingestion
    cleaned_df.to_csv("cleaned_rock_events.csv", index=False)
    
    # Compute and display saturation report
    saturation_df = compute_market_saturation(cleaned_df)
    saturation_df.to_csv("rock_market_saturation.csv", index=False)
    
    print("\n=== CORRECTED MARKET SATURATION SUMMARY (ONS Baseline) ===")
    print(saturation_df.to_string(index=False))

=== STARTING LIVE DATA INGESTION ===
City: London       | Page 1/8 | Fetched: 200 | Total Available: 1533
City: London       | Page 2/8 | Fetched: 200 | Total Available: 1533
City: London       | Page 3/8 | Fetched: 200 | Total Available: 1533
City: London       | Page 4/8 | Fetched: 200 | Total Available: 1533
City: London       | Page 5/8 | Fetched: 200 | Total Available: 1533
⚠️ [Truncation Warning] City 'London' hit 1,000 record cap on page 5.
City: Manchester   | Page 1/3 | Fetched: 200 | Total Available: 548
City: Manchester   | Page 2/3 | Fetched: 200 | Total Available: 548
City: Manchester   | Page 3/3 | Fetched: 148 | Total Available: 548
City: Birmingham   | Page 1/2 | Fetched: 200 | Total Available: 290
City: Birmingham   | Page 2/2 | Fetched: 90 | Total Available: 290
City: Glasgow      | Page 1/4 | Fetched: 200 | Total Available: 626
City: Glasgow      | Page 2/4 | Fetched: 200 | Total Available: 626
City: Glasgow      | Page 3/4 | Fetched: 200 | Total Available: 626
City: